In [36]:
import os
import sqlite3
from typing import Annotated, TypedDict
from langchain_core.tools import tool
from langchain_core.messages import (
    HumanMessage,
    SystemMessage,
    RemoveMessage,
)
from langchain_google_genai import ChatGoogleGenerativeAI

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.sqlite import SqliteSaver

from dotenv import load_dotenv

load_dotenv()


True

In [37]:
SUMMARIZE_AFTER_N_MESSAGES=8

In [38]:
class State(TypedDict):
    messages: Annotated[list, add_messages]
    summary: str  # Running plain-text summary of older turns


In [39]:
def extract_text(content) -> str:
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return " ".join(
            part.get("text", "") for part in content if isinstance(part, dict)
        )
    return str(content)

### Tools

In [40]:
@tool
def get_weather(city: str) -> str:
    """Get the current weather for a given city. Use this when the user asks about weather."""
    # Dummy data — replace with a real API call (e.g. OpenWeatherMap) later
    dummy_data = {
        "chennai": "Sunny, 34°C, humidity 70%",
        "london": "Cloudy, 12°C, light drizzle",
        "new york": "Partly cloudy, 18°C",
    }
    return dummy_data.get(city.lower(), f"Weather data not available for '{city}'.")

In [41]:
tools = [get_weather]

### Nodes

In [42]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.environ["GOOGLE_API_KEY"],
)
llm_with_tools = llm.bind_tools(tools)

# A plain (no tools) model just for summarization calls
summarization_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.environ["GOOGLE_API_KEY"],
    max_output_tokens=256,
)

In [43]:
def chatbot(state: State) -> dict:
    existing_summary = state.get("summary", "")

    sys_msg_content = (
        "You are a helpful, intelligent AI assistant. "
        "Answer general questions using internal knowledge. "
        "Use tools only when strictly necessary."
    )
    if existing_summary:
        sys_msg_content += f"\n\nSummary of earlier conversation:\n{existing_summary}"

    final_messages = [SystemMessage(content=sys_msg_content)] + state["messages"]
    response = llm_with_tools.invoke(final_messages)
    return {"messages": [response]}

In [44]:
tool_node = ToolNode(tools)

In [45]:
def summarize(state: State) -> dict:
    """
    Summarise the oldest messages, store result in `summary`,
    and delete those messages from state via RemoveMessage.
    Always keeps the last 2 messages (most recent exchange) intact.
    """
    messages = state["messages"]
    existing_summary = state.get("summary", "")

    # Build the prompt for the summarization LLM
    if existing_summary:
        summary_prompt = (
        f"Existing summary of the conversation so far:\n{existing_summary}\n\n"
        "Extend the summary by incorporating the NEW messages listed below. "
        "Be concise — output only the updated summary, no preamble."
        )
    else:
        summary_prompt = (
        "Create a concise summary of the conversation below. "
        "Output only the summary, no preamble."
        )

    # Include all messages except the last 2 in the summarization request
    messages_to_summarize = messages[:-2]
    summarization_input = messages_to_summarize + [
    HumanMessage(content=summary_prompt)
    ]

    response = summarization_llm.invoke(summarization_input)
    new_summary = response.content

    # Delete the old messages that were just summarised
    delete_ops = [RemoveMessage(id=m.id) for m in messages_to_summarize]

    return {"summary": new_summary, "messages": delete_ops}

### CONDITIONAL EDGE

In [46]:
def should_summarize(state: State) -> str:
    """
    Called after chatbot (and after tools → chatbot).
    Routes to 'summarize' when message count exceeds the threshold,
    otherwise routes to END.
    """
    if len(state["messages"]) > SUMMARIZE_AFTER_N_MESSAGES:
        return "summarize"
    return END

### Graphs

In [47]:
graph_builder = StateGraph(State)

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", tool_node)
graph_builder.add_node("summarize", summarize)


In [48]:
graph_builder.add_edge(START, "chatbot")

# ✅ Single conditional edge — tools_condition's END is remapped to "summarize"
graph_builder.add_conditional_edges(
    "chatbot",
    tools_condition,  # returns "tools" or "__end__"
    {"tools": "tools", END: "summarize"},  # remap END → summarize check
)

# summarize check routes to "summarize" node or actual END
graph_builder.add_conditional_edges(
    "summarize",  # reuse the node name (rename node to "check_summarize")
    should_summarize,
    {"summarize": "summarize", END: END},
)

graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge("summarize", END)

In [49]:
conn = sqlite3.connect("./SQLDB/state_db.sqlite", check_same_thread=False)
memory = SqliteSaver(conn)
memory.setup()

In [50]:
graph = graph_builder.compile(checkpointer=memory)

In [51]:
config = {"configurable": {"thread_id": "user_4"}}

In [52]:
# %%
user_input = input("Query: ")

for chunk in graph.stream(
    {"messages": [HumanMessage(content=user_input)]},  # type: ignore
    config,  # type: ignore
    stream_mode="values",  # emits full state after every node
):
    chunk["messages"][-1].pretty_print()


================================ Human Message =================================

what are the questions I have asked?
================================== Ai Message ==================================

You have asked me:

1. "how are you?"
2. "what is Contry?"
3. "what are the questions I have asked?"
================================== Ai Message ==================================

You have asked me:

1. "how are you?"
2. "what is Contry?"
3. "what are the questions I have asked?"


In [53]:
# %%
user_input = input("Query: ")

for chunk, metadata in graph.stream(
    {"messages": [HumanMessage(content=user_input)]},  # type: ignore
    config,  # type: ignore
    stream_mode="messages",
):
    print(f"NODE: {metadata['langgraph_node']}")  # type: ignore
    print(f"TYPE: {chunk.__class__.__name__}")
    print(f"CONTENT: {repr(chunk.content)}")  # type: ignore
    print("---")


NODE: chatbot
TYPE: AIMessageChunk
CONTENT: "LangGraph is a library that helps you build applications with language models. It's built on top of LangChain and"
---
NODE: chatbot
TYPE: AIMessageChunk
CONTENT: ' allows you to create more complex, stateful, and multi-actor applications by representing your application as a graph. This means you can define different nodes (which can be anything from a language model call to a tool usage or a custom Python'
---
NODE: chatbot
TYPE: AIMessageChunk
CONTENT: " function) and edges (which define the flow between these nodes) to create intricate workflows.\n\nIt's particularly useful for:\n\n*   **Agents with memory and complex reasoning:** You can design agents that remember past interactions and make decisions based on a"
---
NODE: chatbot
TYPE: AIMessageChunk
CONTENT: ' sequence of steps.\n*   **Multi-agent systems:** You can orchestrate multiple language models or tools to work together in a coordinated fashion.\n*   **Customizable control flo

In [54]:
full_state = graph.get_state(config)  # type: ignore

print(f"Summary: {full_state.values.get('summary', 'None yet')}")
print(f"Messages in state: {len(full_state.values['messages'])}")

for i, msg in enumerate(full_state.values["messages"]):
    print(f"\n[{i}] {msg.__class__.__name__}")
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print(f"  tool_calls : {msg.tool_calls}")
    if hasattr(msg, "name") and msg.name:
        print(f"  tool_name  : {msg.name}")
    print(f"  content    : {extract_text(msg.content) or '(empty - tool dispatch)'}")

Summary: The user asked "how are you?" and then "what is Contry?". The AI provided a definition of a country. The user then asked "what are the questions I have asked?", and the AI listed all three questions the user had posed.
Messages in state: 2

[0] HumanMessage
  content    : what about LangGraph?

[1] AIMessage
  content    : LangGraph is a library that helps you build applications with language models. It's built on top of LangChain and allows you to create more complex, stateful, and multi-actor applications by representing your application as a graph. This means you can define different nodes (which can be anything from a language model call to a tool usage or a custom Python function) and edges (which define the flow between these nodes) to create intricate workflows.

It's particularly useful for:

*   **Agents with memory and complex reasoning:** You can design agents that remember past interactions and make decisions based on a sequence of steps.
*   **Multi-agent systems: